# Урок 16. Объекты сотрудничают: композиция, представление и наследование

В первом уроке ООП мы описали отдельного персонажа и его состояние. Теперь соберём несколько объектов в одну систему. Разберём, как объекты хранят ссылки друг на друга, как защитить допустимое состояние и как задавать удобное текстовое представление экземпляра.

**Результаты урока:** ты сможешь объяснить связь «содержит» и «является», собрать объект из других объектов, защитить простое ограничение состояния, написать `__str__` и `__repr__`, прочитать минимальный пример наследования.

> Запускай ячейки сверху вниз. Перед запуском предсказывай, какой объект изменится и что увидит пользователь.

## 1. Композиция: объект содержит другие объекты

У команды есть участники. Каждый участник — отдельный объект `Player`, а команда хранит список ссылок на них. Это отношение **«команда содержит игроков»**. Оно называется композицией, когда объект отвечает за создание и использование частей своей модели.

Сначала проследи: создание команды не превращает игроков в копии. Команда хранит ссылки на те же объекты.

In [ ]:
class Player:
    def __init__(self, name, score=0):
        self.name = name
        self.score = score

    def add_points(self, amount):
        self.score += amount


class Team:
    def __init__(self, name, players):
        self.name = name
        self.players = players.copy()

    def add_player(self, player):
        self.players.append(player)

    def total_score(self):
        total = 0
        for player in self.players:
            total += player.score
        return total

    def show_roster(self):
        for player in self.players:
            print(player.name, player.score)


mira = Player("Мира", 5)
leo = Player("Лев", 3)
team = Team("Комета", [mira, leo])
print(team.total_score())
mira.add_points(2)
print(team.total_score())

`players.copy()` создаёт отдельный контейнер списка, поэтому добавление игрока в исходный список после создания команды не меняет состав команды. Но сами объекты `Player` не копируются: изменение `mira.score` видно команде, поскольку `team.players` хранит ссылку на тот же экземпляр.

**Проверь.** Создай новый список `players`, передай его в `Team`, затем добавь в исходный список ещё одного игрока. После этого вызови `show_roster()`. Объясни результат.

## 2. Состояние объекта и инвариант

**Инвариант** — правило, которое объект должен сохранять после каждого своего действия. Для кошелька это может быть «баланс не бывает отрицательным». Если каждый вызов сначала проверяет условие, некорректное изменение не попадёт в состояние.

In [ ]:
class Wallet:
    def __init__(self, owner, balance=0):
        if balance < 0:
            raise ValueError("Начальный баланс не может быть отрицательным")
        self.owner = owner
        self.balance = balance

    def deposit(self, amount):
        if amount <= 0:
            return False
        self.balance += amount
        return True

    def withdraw(self, amount):
        if amount <= 0 or amount > self.balance:
            return False
        self.balance -= amount
        return True


wallet = Wallet("Мира", 100)
print(wallet.withdraw(30), wallet.balance)
print(wallet.withdraw(100), wallet.balance)
print(wallet.deposit(20), wallet.balance)

Снятие проверяет сумму **до** изменения баланса. Если денег недостаточно, метод возвращает `False`, а состояние остаётся прежним. Это помогает объекту самому отвечать за правила своих данных.

**Важно:** пример использует `ValueError`, который уже разобран в уроке об ошибках. Для операций кошелька мы выбрали явный результат `True`/`False`, чтобы вызывающий код мог сообщить пользователю об отказе.

## 3. Понятный вывод: `__str__` и `__repr__`

`print(object)` обычно вызывает `__str__`: человекочитаемое представление для пользователя. `repr(object)` вызывает `__repr__`: более точное представление для разработчика при отладке. Если написать только `__repr__`, Python использует его и для `str`.

Оба метода возвращают строку; они не печатают её сами.

In [ ]:
class Book:
    def __init__(self, title, pages):
        self.title = title
        self.pages = pages

    def __str__(self):
        return f"«{self.title}» — {self.pages} стр."

    def __repr__(self):
        return f"Book(title={self.title!r}, pages={self.pages!r})"


book = Book("Остров сокровищ", 320)
print(book)
print(repr(book))
print([book])

В списке Python показывает `repr()` элементов: это удобно, когда нужно понять, какие данные хранятся внутри. Формат `!r` в f-строке вставляет значение через `repr`, например добавляет кавычки вокруг строки.

**Мини-практика.** Добавь `__str__` классу `Player`, чтобы он возвращал `Имя: N очков`. Затем добавь `__repr__`, содержащий название класса и аргументы, нужные для описания игрока.

## 4. Наследование: отношение «является»

Наследование подходит, когда новый класс **является разновидностью** существующего и может использовать его поведение. Дочерний класс наследует методы базового класса и может переопределить нужный метод. Вызов `super().__init__(...)` запускает инициализацию базового класса.

Если объект лишь **содержит** другой объект, обычно яснее композиция из раздела 1. Не создавай базовый класс только ради повторного использования одной строки.

In [ ]:
class Message:
    def __init__(self, text):
        self.text = text

    def deliver(self):
        return f"Сообщение: {self.text}"


class EmailMessage(Message):
    def __init__(self, text, address):
        super().__init__(text)
        self.address = address

    def deliver(self):
        return f"Письмо для {self.address}: {self.text}"


message = Message("Встреча в 15:00")
email = EmailMessage("Встреча в 15:00", "mira@example.com")
print(message.deliver())
print(email.deliver())

`EmailMessage` — разновидность `Message`, поэтому наследует его атрибут `text`. Класс задаёт собственный `deliver()`, то есть **переопределяет** метод. Вызов выбирает метод по фактическому типу объекта.

**Короткий прогноз.** Что напечатает `print(isinstance(email, Message))`? Запусти после того, как ответишь.

In [ ]:
print(isinstance(email, Message))

### Почему тестовый класс тоже наследует

В следующем уроке тестовый класс будет записан как `class TestSomething(unittest.TestCase):`. Это тот же синтаксис наследования: `TestSomething` — специализированный класс теста. Он получает готовые методы вроде `assertEqual()` от `unittest.TestCase` и добавляет собственные методы с именами `test_...`.

Знать устройство `unittest` заранее не требуется: достаточно понимать, что в скобках указана база, от которой класс наследует поведение. Пример в конце этого урока показывает наследование на собственных классах.

## Проверь себя

- Почему копия списка команды не копирует самих игроков?
- Как объект защищает инвариант баланса?
- Когда вызывается `__str__`, а когда `__repr__`?
- Что делает `super().__init__()`?
- В каком случае композиция проще наследования?

## Домашняя работа

Все задания находятся в этом блокноте. Обязательны задачи 1–3 и задача 4 на повторение. Для функций и методов в таблицах показаны аргументы и возвращаемое значение или новое состояние; ввод с клавиатуры не нужен. В примерах каждый вызов выполняй отдельно.

### ДЗ-16.1. Состав команды · обязательная

**Сюжет.** Тренер ведёт список участников и хочет получить общий счёт команды.

**Условие.** Создайте `Player(name, score)` и `Team(name, players)`. Метод `total_score()` возвращает сумму очков. Конструктор команды сохраняет отдельную копию контейнера списка, но не копирует объекты игроков.

**Формат данных**

| Входные данные | Выходные данные |
|---|---|
| Строка имени команды и список объектов `Player`. | Объект `Team`; `total_score()` возвращает целое число. |

**Ограничения.** До 20 игроков; очки неотрицательные целые. Для пустой команды сумма равна 0.

**Примеры**

| Входные данные | Выходные данные |
|---|---|
| <pre>team = Team("Комета", [Player("Мира", 5), Player("Лев", 3)])<br>team.total_score()</pre> | <pre>8</pre> |
| <pre>Team("Новая", []).total_score()</pre> | <pre>0</pre> |

**Источник.** Авторская задача курса.

### ДЗ-16.2. Безопасный кошелёк · обязательная

**Сюжет.** Приложение не должно разрешать снять больше денег, чем есть на счёте.

**Условие.** Создайте `Wallet(balance)`, метод `withdraw(amount)` возвращает `True` и списывает сумму при успешном снятии. Иначе возвращает `False`, не меняя баланс.

**Формат данных**

| Входные данные | Выходные данные |
|---|---|
| Начальный баланс и последовательность сумм для снятия. | Для каждого снятия — результат `True`/`False` и текущий баланс. |

**Ограничения.** Начальный баланс и суммы — неотрицательные целые; сумма 0 считается недопустимой.

**Примеры**

| Входные данные | Выходные данные |
|---|---|
| <pre>Wallet(50); withdraw(20); withdraw(40)</pre> | <pre>True, 30<br>False, 30</pre> |
| <pre>Wallet(0); withdraw(1)</pre> | <pre>False, 0</pre> |

**Источник.** Авторская задача курса.

### ДЗ-16.3. Карточка книги · обязательная

**Сюжет.** Библиотекарь хочет видеть короткую карточку книги и подробную запись для отладки.

**Условие.** Для `Book(title, pages)` реализуйте `__str__`, возвращающий `Название — N стр.`, и `__repr__`, возвращающий строку вида `Book(title='...', pages=N)`. Не вызывайте `print()` внутри специальных методов.

**Формат данных**

| Входные данные | Выходные данные |
|---|---|
| Название книги и целое число страниц. | `str(book)` — короткая карточка; `repr(book)` — подробное представление. |

**Ограничения.** Название непустое, страницы положительные; в названии нет кавычки `'`.

**Примеры**

| Входные данные | Выходные данные |
|---|---|
| <pre>book = Book("Маугли", 180)<br>str(book)</pre> | <pre>Маугли — 180 стр.</pre> |
| <pre>repr(book)</pre> | <pre>Book(title='Маугли', pages=180)</pre> |

**Источник.** Авторская задача курса.

### ДЗ-16.4. Уведомления · повторение

**Сюжет.** Сервис отправляет обычные сообщения и письма с адресом получателя.

**Условие.** Реализуйте базовый класс `Message(text)` с методом `deliver()`, затем `EmailMessage(text, address)`, который наследуется от `Message` и переопределяет `deliver()`. Используйте `super().__init__(text)`.

**Формат данных**

| Входные данные | Выходные данные |
|---|---|
| Текст сообщения; для письма — также адрес. | `Message.deliver()` возвращает `Сообщение: ТЕКСТ`; `EmailMessage.deliver()` — `Письмо для АДРЕС: ТЕКСТ`. |

**Ограничения.** Текст и адрес непустые строки.

**Примеры**

| Входные данные | Выходные данные |
|---|---|
| <pre>Message("До встречи").deliver()</pre> | <pre>Сообщение: До встречи</pre> |
| <pre>EmailMessage("До встречи", "a@example.org").deliver()</pre> | <pre>Письмо для a@example.org: До встречи</pre> |

**Повторение:** классы, `self`, `__init__`, аргументы методов, строки и `return`.

**Источник.** Авторская задача курса.